# `clonealign` on iscc data — an R notebook

This notebook runs the **real [`clonealign`](https://github.com/kieranrcampbell/clonealign)**
(Campbell et al. 2019) on data simulated by `iscc`, and scores it against ground truth the tool never
sees. It is an **R notebook** — kernel `R (iscc-clonealign)` — so the code below is the R you would
write yourself, not a Python wrapper around it.

It contains no simulation. The dataset was generated once by
`python validation/make_analysis_data.py` and written to `analysis_data/clonealign/`:

| file | what it is | who sees it |
|---|---|---|
| `Y.csv.gz` | scRNA counts, cells × genes | the tool |
| `L.csv.gz` | copy number, genes × clones | the tool |
| `truth.csv` | each cell's true clone | **scoring only** |

Separating generation from analysis is what lets this be an R notebook at all: a notebook has one
kernel, and growing the tumour is Python. It also keeps the benchmark honest — the analysis side
cannot reach anything the tool would not have.

**The tumour.** A realistic breach-gated ductal field (grid 96, 5 glands, 6,000 genes), not a toy rig.
It is WGD+, and its clones are near-neighbours in copy-number space — see `meta.json` below.

In [1]:
# reticulate must be pointed at THIS env's Python, or it searches a cached uv interpreter and
# reports "Valid installation of TensorFlow not found" — clonealign's backend is TensorFlow via
# reticulate, so this preamble is load-bearing, not boilerplate.
local({
  env_py <- file.path(dirname(dirname(R.home())), "bin", "python")
  if (file.exists(env_py)) Sys.setenv(RETICULATE_PYTHON = env_py)
})

suppressWarnings(suppressMessages({
  library(clonealign)
  library(tensorflow)
  library(jsonlite)
}))

data_dir <- file.path("..", "analysis_data", "clonealign")
stopifnot(dir.exists(data_dir))

Y <- as.matrix(read.csv(file.path(data_dir, "Y.csv.gz"), row.names = 1, check.names = FALSE))
L <- as.matrix(read.csv(file.path(data_dir, "L.csv.gz"), row.names = 1, check.names = FALSE))
meta <- fromJSON(file.path(data_dir, "meta.json"))

cat(sprintf("Y: %d cells x %d genes\nL: %d genes x %d clones\n",
            nrow(Y), ncol(Y), nrow(L), ncol(L)))
cat(sprintf("clone sizes: %s\n", paste(meta$clone_sizes, collapse = ", ")))
cat(sprintf("baselines — chance %.2f, majority %.2f\n",
            meta$chance_baseline, meta$majority_baseline))
cat("\nclone copy-number consensus (rows = clones, cols = segments):\n")
print(meta$cn_consensus)

Y: 1527 cells x 6000 genes
L: 6000 genes x 3 clones


clone sizes: 1062, 211, 254


baselines — chance 0.33, majority 0.70



clone copy-number consensus (rows = clones, cols = segments):


     [,1] [,2] [,3] [,4] [,5] [,6] [,7] [,8] [,9] [,10] [,11] [,12]
[1,]    4    4    4    4    5    4    4    7    4     4     4     4
[2,]    4    4    4    4    6    4    4    7    4     4     4     4
[3,]    4    4    4    4    6    4    3    7    4     4     4     4


## Why this is a hard problem

Look at the consensus above: the clones differ in only **one or two of the twelve segments**. The
tumour is whole-genome doubled, so almost every segment sits at copy number 4 in every clone, and the
divergence is concentrated in a couple of places.

`clonealign` is a **dosage** model — it assumes a gene's expression scales with the number of copies
the clone carries. When clones share copy number nearly everywhere, that signal is thin. This is the
realistic case; the toy substrate these benchmarks used previously had far more divergent clones and
made the task look easier than it is.

Note the bar to clear is the **majority baseline** (always guess the biggest clone), not chance.

In [2]:
# Align genes across the two modalities, then fit.
common <- intersect(colnames(Y), rownames(L))
Yc <- Y[, common, drop = FALSE]; storage.mode(Yc) <- "double"
Lc <- L[common, , drop = FALSE]; storage.mode(Lc) <- "double"

set.seed(1); tf$compat$v1$set_random_seed(1L)

# clonealign's own multi-restart wrapper. The variational objective is sensitive to initialisation,
# so a single fit can settle in a poor local optimum; this is the package's intended entry point.
fit <- run_clonealign(Yc, Lc,
                      initial_shrinks = c(0, 5, 10),
                      n_repeats = 3,
                      max_iter = 200,
                      print_elbos = FALSE,
                      verbose = FALSE)

probs <- fit$ml_params$clone_probs
colnames(probs) <- colnames(Lc); rownames(probs) <- rownames(Yc)
cat(sprintf("fitted: %d cells assigned over %d clones\n", nrow(probs), ncol(probs)))
table(fit$clone)

fitted: 1527 cells assigned over 3 clones



    clone0     clone1     clone2 unassigned 
       584        202        169        572 

## Scoring against ground truth

Only now do we open `truth.csv`. Predicted clone labels are arbitrary, so we take the best
permutation of predicted-to-true labels (with three clones there are only six, so we enumerate them
rather than reaching for the Hungarian algorithm).

In [3]:
truth <- read.csv(file.path(data_dir, "truth.csv"), stringsAsFactors = FALSE)
true_clone <- truth$true_clone[match(rownames(probs), truth$cell)]
pred <- max.col(probs) - 1L                      # 0-based, to match the truth file

# best label permutation (3 clones -> 6 permutations, so enumerate rather than use Hungarian)
K <- ncol(probs)
perms <- as.matrix(expand.grid(rep(list(0:(K - 1)), K)))
perms <- perms[apply(perms, 1, function(r) length(unique(r)) == K), , drop = FALSE]
best <- which.max(apply(perms, 1, function(p) mean(p[pred + 1L] == true_clone)))
acc  <- mean(perms[best, ][pred + 1L] == true_clone)
maj  <- max(table(true_clone)) / length(true_clone)

# ACCURACY ALONE IS MISLEADING HERE: the clones are 1062/211/254, so "always guess the biggest"
# already scores ~0.70. Rank-based measures say whether the tool found signal at all, independent
# of that imbalance.
adj_rand <- function(a, b) {                     # adjusted Rand index, from the contingency table
  tab <- table(a, b); n <- length(a)
  ch2 <- function(x) sum(x * (x - 1) / 2)
  idx <- ch2(tab); ea <- ch2(rowSums(tab)); eb <- ch2(colSums(tab)); tot <- n * (n - 1) / 2
  (idx - ea * eb / tot) / (0.5 * (ea + eb) - ea * eb / tot)
}
auc1 <- function(score, pos) {                   # one-vs-rest AUC (Mann-Whitney)
  r <- rank(score); np <- sum(pos); nn <- sum(!pos)
  if (np == 0 || nn == 0) return(NA_real_)
  (sum(r[pos]) - np * (np + 1) / 2) / (np * nn)
}
mapped_true <- match(true_clone, perms[best, ]) - 1L   # true labels in predicted-column space
aucs <- sapply(seq_len(K), function(k) auc1(probs[, k], mapped_true == (k - 1L)))

cat("clonealign vs iscc ground truth\n")
cat(sprintf("  accuracy %.2f   (chance %.2f, majority %.2f)  -> %s the majority baseline\n",
            acc, 1 / K, maj, ifelse(acc > maj + 0.02, "ABOVE", "AT OR BELOW")))
cat(sprintf("  ARI      %.2f   (0 = no better than random grouping)\n", adj_rand(pred, true_clone)))
cat(sprintf("  mean AUC %.2f   per clone: %s\n", mean(aucs, na.rm = TRUE),
            paste(sprintf("%.2f", aucs), collapse = ", ")))
cat("\n  AUC above 0.5 with accuracy at the majority baseline means the tool DID find signal,\n")
cat("  but not enough to win hard assignments against a clone holding 70% of the cells.\n")

clonealign vs iscc ground truth


  accuracy 0.67   (chance 0.33, majority 0.70)  -> AT OR BELOW the majority baseline


  ARI      0.26   (0 = no better than random grouping)


  mean AUC 0.81   per clone: 0.82, 0.77, 0.84



  AUC above 0.5 with accuracy at the majority baseline means the tool DID find signal,


  but not enough to win hard assignments against a clone holding 70% of the cells.


## Reading the result

If the accuracy sits at the majority baseline, `clonealign` has not found the clone structure — and
on this tumour that is the expected outcome rather than a failure of the tool. Copy-number dosage is
almost uninformative when clones share copy number in ten of twelve segments.

The companion notebook `combining_scdna_scrna.ipynb` shows what *does* separate these clones: the
**allele** layer. Two clones can both sit at total copy number 4 while differing in their allelic
composition (4+0 versus 2+2), and B-allele frequency sees a difference that total copy number cannot.

That contrast is the point. A benchmark on a substrate where every tool succeeds tells you nothing
about which modality carries the signal.